# Big Data Analytics — Progetto: Apache Spark RDD

## Obiettivo

Questo notebook esplora due funzionalità fondamentali di Apache Spark attraverso
il dataset pubblico **Divvy Bikes** (Chicago bike-sharing system).

### Parte 1  RDD e In-Memory Computing
Implementiamo l'algoritmo **K-Means clustering** manualmente con RDD puri,
applicato alle coordinate geografiche delle stazioni Divvy.

L'algoritmo viene eseguito in due modalità:
- **Senza `persist()`**: ad ogni iterazione Spark ricalcola l'RDD risalendo
  il lineage dal file sorgente, comportamento analogo al classico MapReduce,
  dove i dati intermedi vengono riletti da disco ad ogni passo.
- **Con `persist()`**: l'RDD viene materializzato in RAM alla prima computazione
  e riutilizzato nelle iterazioni successive, eliminando l'I/O ripetuto.

Il confronto mostra quantitativamente il vantaggio dell'in-memory computing
per algoritmi iterativi, che rappresentano il caso d'uso in cui Spark supera
più nettamente il paradigma MapReduce tradizionale.

### Parte 2 — Structured Streaming
Estendiamo l'elaborazione al caso **online**: simuliamo un flusso continuo di
nuovi trip Divvy e usiamo Spark Structured Streaming  e DStream per aggiornare in
tempo reale le aggregazioni per stazione (trip attivi, stazioni più usate).

Questo scenario evidenzia i limiti dei DBMS relazionali classici per dati
in movimento e motiva l'uso di un framework come Spark anche per workload
di tipo streaming.

In [6]:
import pyspark
print(pyspark.__version__)

3.5.3


Inizio parte 1
- Import

In [7]:
from pyspark import SparkContext, SparkConf
import os, time, zipfile
import numpy as np
import matplotlib.pyplot as plt

Download del dataset

In [1]:
# URL per il download
URL  = "https://divvy-tripdata.s3.amazonaws.com/202409-divvy-tripdata.zip"
CSV_PATH = "202409-divvy-tripdata.csv"

Setup Spark

In [9]:
conf = SparkConf() \
    .setAppName("DivvyKMeans") \
    .setMaster("local[*]") \
    .set("spark.driver.memory", "2g")

sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")
print("Spark avviato:", sc.version)
print("UI disponibile su: http://localhost:4040")

26/06/08 16:07:28 WARN Utils: Your hostname, pietro-NBD-WXX9 resolves to a loopback address: 127.0.1.1; using 192.168.1.87 instead (on interface wlp0s20f3)
26/06/08 16:07:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 16:07:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark avviato: 3.5.3
UI disponibile su: http://localhost:4040


Lettura del csv
- parse_coords legge le righe e prende solo il campo "latitude, longitude"

In particolare start_lat e start_lng sono le coordinate della stazione di partenza del viaggio, il punto dove l'utente ha preso la bici.<br>
Per il K-Means l'idea è raggruppare i punti di partenza in cluster geografici per capire quali zone di Chicago hanno più domanda. <br>
Ogni punto nell'RDD rappresenta un viaggio, quindi le zone con più punti concentrati sono quelle con più utilizzo.

In [10]:
raw_rdd = sc.textFile(CSV_PATH)
header  = raw_rdd.first()

def parse_coords(line):
    fields = line.split(",")
    try:
        lat = float(fields[8])
        lng = float(fields[9])
        if 41.6 <= lat <= 42.1 and -87.9 <= lng <= -87.5:  #  filtro rumore: bounding box Chicago
            return (lat, lng)
    except:
        pass
    return None

coords_rdd = raw_rdd \
    .filter(lambda line: line != header) \
    .map(parse_coords) \
    .filter(lambda x: x is not None)

n_points = coords_rdd.count()
print(f"Punti validi: {n_points:,}")

Punti validi: 821,275


Funzioni per implementazione del K-means

- dist_sq(a, b)
Calcola la distanza euclidea
- assign_cluster(point, centroids)
Dato un punto e la lista dei centroidi, restituisce l'indice del centroide più vicino. 
- kmeans computa iterativamente i centroidi fino a convergenza, il flag "use_cache" sarà usato per simulare RDD+in_memory (come fa SPARK) e confrontandolo con la versione precedente map/reduce che non supporta in memory.

In [11]:
def dist_sq(a, b):
    return (a[0]-b[0])**2 + (a[1]-b[1])**2

def assign_cluster(point, centroids):
    return min(range(len(centroids)), key=lambda i: dist_sq(point, centroids[i]))

''' 
MAP: ogni punto viene trasformato in (cluster_id, (lat, lng, 1)) dove 1 è il contatore per calcolare la media dopo.
REDUCE: i punti dello stesso cluster vengono aggregati sommando lat, lng e contatore: (sum_lat, sum_lng, count).

In sostanza bisogna computare iterativamente una MEDIA per il calcolo del centroide, quindi per aggregare è necessario avere somma e 
contatore. Questo perchè la media NON è associativa
'''
def kmeans(rdd, k=10, max_iter=20, use_cache=False, label=""):
    centroids = rdd.takeSample(False, k, seed=42)       #primi k centroidi scelti casualmente
    
    if use_cache:
        rdd.persist()
        _ = rdd.count()  # forza materializzazione prima di misurare
    
    iter_times = []

    for i in range(max_iter):
        t0 = time.time()

        bc = sc.broadcast(centroids) #invia la lista dei centroidi a tutti i worker

        # MAP: ogni punto → (cluster_id, (lat, lng, 1))
        # REDUCE: somma per cluster → nuovo centroide 
        new_centroids_rdd = (
            rdd
            .map(lambda p: (assign_cluster(p, bc.value), (p[0], p[1], 1)))      #ad ogni punto assegni il cluster più vicino e trasformi in (cluster_id, (lat, lng, 1))
            .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))   #reduceByKey per sommare lat, lng e contatore per ogni cluster_id
            .map(lambda kv: (kv[0], (kv[1][0]/kv[1][2], kv[1][1]/kv[1][2]))) #calcola il nuovo centroide come media: (sum_lat/count, sum_lng/count
        )

        new_map   = new_centroids_rdd.collectAsMap()
        new_cents = [new_map.get(j, centroids[j]) for j in range(k)]

        elapsed = time.time() - t0
        iter_times.append(elapsed)

        shift = max(dist_sq(centroids[j], new_cents[j]) for j in range(k))
        centroids = new_cents
        bc.unpersist()

        print(f"  [{label}] iter {i+1:2d} | {elapsed:.3f}s | shift: {shift:.2e}")

        if shift < 1e-9:        #convergenza se il delta tra i centroidi di questa it e la precedente è molto piccolo
            print(f"  Convergenza raggiunta all'iterazione {i+1}")
            break

    if use_cache:
        rdd.unpersist()

    return centroids, iter_times

Esecuzione del K-means :
- Map/reduce "nativo" senza in-memory  (nativo nel senso prima dell'arrivo di Spark e della tecnologia in-memory)
- Spark

In [12]:
print("=== SENZA persist() — equivalente MapReduce ===")
cents_no, times_no = kmeans(coords_rdd, k=15, max_iter=25, use_cache=False, label="no-cache")

=== SENZA persist() — equivalente MapReduce ===


  [no-cache] iter  1 | 2.596s | shift: 5.48e-04


  [no-cache] iter  2 | 2.811s | shift: 2.18e-04


  [no-cache] iter  3 | 2.443s | shift: 8.68e-05


  [no-cache] iter  4 | 2.395s | shift: 4.87e-05


...

  [no-cache] iter 21 | 2.365s | shift: 4.69e-06


  [no-cache] iter 22 | 2.350s | shift: 3.86e-06


  [no-cache] iter 23 | 2.839s | shift: 3.19e-06


  [no-cache] iter 24 | 2.286s | shift: 5.02e-06


  [no-cache] iter 25 | 2.721s | shift: 3.83e-06


In [13]:

print("\n=== CON persist() — in-memory computing ===")
cents_ok, times_ok = kmeans(coords_rdd, k=15, max_iter=25, use_cache=True,  label="cached")

print(f"\nTotale senza cache : {sum(times_no):.2f}s")
print(f"Totale con cache   : {sum(times_ok):.2f}s")
print(f"Speedup            : {sum(times_no)/sum(times_ok):.2f}x")


=== CON persist() — in-memory computing ===


  [cached] iter  1 | 1.731s | shift: 5.48e-04


  [cached] iter  2 | 1.741s | shift: 2.18e-04


  [cached] iter  3 | 1.718s | shift: 8.68e-05


  [cached] iter  4 | 1.749s | shift: 4.87e-05


...

  [cached] iter 21 | 1.818s | shift: 4.69e-06


  [cached] iter 22 | 1.791s | shift: 3.86e-06


  [cached] iter 23 | 1.917s | shift: 3.19e-06


  [cached] iter 24 | 1.827s | shift: 5.02e-06


  [cached] iter 25 | 1.974s | shift: 3.83e-06

Totale senza cache : 61.55s
Totale con cache   : 45.75s
Speedup            : 1.35x


Provo ad usare come benchmark un algoritmo meno cpu-bound:
-  calcolo iterativamente semplicemente un count
- nella versione in memory dalla 2* it in poi uso in memory

In [14]:
import time

# Versione senza cache: ogni count() rilegge e riesegue tutto il lineage
print("=== Benchmark lineage: SENZA persist() ===")
times_no = []
for i in range(10):
    t0 = time.time()
    _ = coords_rdd.count()
    times_no.append(time.time() - t0)
    print(f"  run {i+1} | {times_no[-1]:.3f}s")

# Versione con cache: prima count() materializza, le successive leggono da RAM
print("\n=== Benchmark lineage: CON persist() ===")
coords_rdd.persist()
times_cache = []
for i in range(10):
    t0 = time.time()
    _ = coords_rdd.count()
    times_cache.append(time.time() - t0)
    print(f"  run {i+1} | {times_cache[-1]:.3f}s")
coords_rdd.unpersist()

print(f"\nMedia senza cache : {sum(times_no[1:])/len(times_no[1:]):.3f}s")
print(f"Media con cache   : {sum(times_cache[1:])/len(times_cache[1:]):.3f}s")
print(f"Speedup           : {sum(times_no[1:])/sum(times_cache[1:]):.2f}x")

=== Benchmark lineage: SENZA persist() ===


  run 1 | 0.865s


  run 2 | 0.910s


  run 3 | 0.752s


  run 4 | 0.793s


...

  run 7 | 0.713s


  run 8 | 0.735s


  run 9 | 0.775s


  run 10 | 0.748s

=== Benchmark lineage: CON persist() ===


  run 1 | 1.000s
  run 2 | 0.174s
  run 3 | 0.179s
  run 4 | 0.151s
  run 5 | 0.176s
  run 6 | 0.164s
  run 7 | 0.167s
  run 8 | 0.172s
  run 9 | 0.165s
  run 10 | 0.159s

Media senza cache : 0.768s
Media con cache   : 0.168s
Speedup           : 4.58x


## Osservazioni sui risultati

I risultati  mostrano speedup significativi e coerenti :
**1.35x per il K-Means** e **4.58x per il benchmark diretto sul lineage**.

### K-Means — speedup ~1.35x

Lo speedup è reale ma moderato. Il motivo è che il K-Means è un workload
**CPU-bound**: la parte dominante di ogni iterazione è il calcolo di
`assign_cluster` eseguito in Python puro per ogni punto dell'RDD. Questo
overhead  è presente sia con che senza
`persist()`, e riduce il peso relativo del risparmio di I/O.


### Benchmark lineage — speedup ~4.58x

Il benchmark diretto sul `count()` ripetuto isola invece un workload
**memory-bound**: nessun calcolo pesante, solo lettura e parsing dei dati.
Qui `persist()` elimina quasi interamente il lavoro ripetuto — la prima
esecuzione materializza l'RDD in RAM (1.0s), tutte le successive lo leggono
direttamente dalla memoria (0.16s), con uno speedup di quasi 5x.

Questo è il caso d'uso ideale per l'in-memory computing: pipeline in cui
gli stessi dati vengono letti e processati molte volte, senza trasformazioni
computazionalmente intensive tra una lettura e l'altra.


### Ulteriori miglioramenti attesi su cluster distribuito

I risultati attuali sono ottenuti in modalità `local[*]` su singola macchina.
Su un vero cluster distribuito i vantaggi di `persist()` sarebbero ancora
più marcati per due ragioni:

- **Shuffle di rete**: senza cache ogni iterazione richiederebbe il
  trasferimento dei dati tra nodi via rete, che è ordini di grandezza più
  lento della RAM locale.
- **Dataset oltre la RAM**: con dataset da decine o centinaia di GB che non
  entrano nella memoria di un singolo nodo, `persist()`  diventa indispensabile per evitare la rilettura da
  storage distribuito su ad esempio un HDFS ad ogni iterazione.



## Parte 2 — Spark Streaming (DStream API)

Spark Streaming consente di elaborare dati che arrivano in modo continuo attraverso una sequenza di micro-batch. In questa esercitazione viene utilizzata la **DStream API**, il modello storico di Spark Streaming, basato su una sequenza temporale di RDD.

Ogni intervallo temporale genera un nuovo RDD contenente i dati arrivati nel periodo considerato. Le trasformazioni (`map`, `filter`, `reduceByKey`) vengono quindi applicate a ciascun RDD del flusso.

### Architettura

Il flusso continuo viene suddiviso in micro-batch discreti, ciascuno rappresentato da un RDD:

```text
flusso continuo  →  [RDD_t0] [RDD_t1] [RDD_t2] ...
                        ↑        ↑        ↑
                     batch_0  batch_1  batch_2
                      (5s)      (5s)      (5s)

```

### Simulazione del flusso

Simuliamo il flusso con la strategia **directory watching**: un thread in
background copia porzioni del dataset Divvy in una cartella ogni 5 secondi.


Attraverso textFileStream(), Spark controlla continuamente la directory e genera un nuovo RDD ogni volta che vengono rilevati nuovi file. Ogni RDD viene quindi elaborato dalla pipeline di trasformazioni e aggregazioni definita dall'applicazione.

### Elaborazione del flusso

Per ogni micro-batch vengono eseguiti i seguenti passaggi:

- Lettura dei nuovi file comparsi nella directory monitorata.
- Parsing delle righe CSV ed estrazione delle informazioni rilevanti.
- Filtraggio delle corse appartenenti all'area geografica di Chicago.
- Associazione della coppia (stazione, 1) ad ogni corsa valida.
- Aggregazione tramite reduceByKey() per ottenere il numero di corse per stazione.
- Stampa delle stazioni più attive nel batch corrente.


## Elaborazione a finestra

Oltre alle statistiche del singolo batch, viene mantenuta una finestra temporale scorrevole di 15 secondi aggiornata ogni 5 secondi mediante reduceByKeyAndWindow().

### Perché non un DBMS relazionale

Un DBMS relazionale tradizionale è progettato principalmente per dati persistenti e interrogazioni su dataset statici.<br>
Non è progettato per dati in movimento: per sapere cosa
è cambiato bisogna interrogarlo esplicitamente, simulando lo streaming
con polling continuo. Questo introduce latenza, spreca risorse su query
ripetute che spesso non trovano nuovi dati, e non **scala orizzontalmente**
quando il volume aumenta. Le aggregazioni su finestre temporali scorrevoli
richiederebbero query complesse con self-join o **indici costosi da mantenere aggiornati ad ogni insert** . Spark Structured Streaming gestisce tutto questo
nativamente.



PER FARE PARTIRE LA SIMULAZIONE:
- eseguire la cella "Per pulire le cartelle di streaming" 
- eseguire fino a "Per avviare il thread" (aspettare prima di avviarlo), eseguire il codice sotto fino all'avvio del monitoraggio usando Dstream o Structered stream
- far partire il thread producer

In [15]:
from pyspark.streaming import StreamingContext
import threading, shutil, time, os

In [76]:
''' 
    Setup per simulare un flusso di dati:
'''

STREAM_DIR   = "./stream_input"
CHECKPOINT   = "./stream_checkpoint"
CHUNK_SIZE   = 2000   # righe per batch
BATCH_INTERVAL = 5    # secondi

# Pulisci da run precedenti
for d in [STREAM_DIR, CHECKPOINT]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

# Carica tutte le righe del CSV (senza header)
with open(CSV_PATH, "r", encoding="utf-8") as f:
    all_lines = f.readlines()
header_line = all_lines[0]
data_lines  = all_lines[1:]

print(f"Righe totali nel dataset: {len(data_lines):,}")
print(f"Chunk size: {CHUNK_SIZE} righe → ~{len(data_lines)//CHUNK_SIZE} batch disponibili")

Righe totali nel dataset: 821,276
Chunk size: 2000 righe → ~410 batch disponibili


Definizione del corpo del thread producer di chunks
- ogni "interval" time produce un nuovo csv in una cartella che viene monitarata da spark


In [77]:
producer_running = threading.Event()
producer_running.set()

def producer_thread(lines, chunk_size, output_dir, interval, n_chunks=15):
    """
    Simula un flusso di dati: ogni `interval` secondi scrive un nuovo
    file CSV nella directory monitorata da Spark Streaming.
    """
    for i in range(n_chunks):
        if not producer_running.is_set():
            break
        
        chunk = lines[i * chunk_size : (i + 1) * chunk_size]
        if not chunk:
            break
        
        filepath = os.path.join(output_dir, f"chunk_{i:04d}.csv")
        with open(filepath, "w", encoding="utf-8") as f:
            f.writelines(chunk)
        
        print(f"  [producer] chunk {i+1:02d} scritto → {len(chunk)} righe")
        time.sleep(interval)
    
    print("  [producer] completato.")


Per avviare il thread

In [ ]:

# Avvia il produttore in background
t = threading.Thread(
    target=producer_thread,
    args=(data_lines, CHUNK_SIZE, STREAM_DIR, BATCH_INTERVAL, 15),
    daemon=True
)
t.start()

  [producer] chunk 01 scritto → 2000 righe
  [producer] chunk 02 scritto → 2000 righe


In [66]:
# Se si vuole farlo terminare prima del completamento:
producer_running.clear()

Uso delle API di SPARK per processing in streaming. <br>

Lancio di 2 pipeline streaming:
- trip_counts sono le informazioni del batch corrente
- windowed sono le informazioni aggregate con finestra di 15s

In [ ]:
# Crea StreamingContext con batch interval di 5 secondi
ssc = StreamingContext(sc, BATCH_INTERVAL)
ssc.checkpoint(CHECKPOINT)

# DStream: monitora la directory, ogni nuovo file → un RDD
lines_dstream = ssc.textFileStream(STREAM_DIR)

# --- Parsing: stessa logica della Parte 1 ---
def parse_stream_line(line):
    fields = line.split(",")
    try:
        station = fields[4].strip().strip('"')   # start_station_name
        lat     = float(fields[8])
        lng     = float(fields[9])
        if 41.6 <= lat <= 42.1 and -87.9 <= lng <= -87.5 and station:
            return (station, 1)
    except:
        pass
    return None

# --- Pipeline per batch corrente ---
# Per ogni micro-batch: conta i trip per stazione di partenza
trip_counts = (
    lines_dstream
    .map(parse_stream_line)
    .filter(lambda x: x is not None)
    .reduceByKey(lambda a, b: a + b)
)

# Top 5 stazioni più attive nel batch corrente
def print_top_stations(rdd):
    if rdd.isEmpty():
        print("  [batch] nessun dato")
        return
    top = rdd.takeOrdered(5, key=lambda x: -x[1])
    batch_time = time.strftime("%H:%M:%S")
    total = rdd.map(lambda x: x[1]).sum()
    print(f"\n  [{batch_time}] Batch — {total} trip processati")
    for station, count in top:
        bar = "█" * min(count // 10, 40)
        print(f"    {station[:35]:<35} {count:4d}  {bar}")

trip_counts.foreachRDD(print_top_stations)

# --- Pipeline con finestra temporale (ultimi 3 batch = 15 secondi) ---
windowed = trip_counts.reduceByKeyAndWindow(
    lambda a, b: a + b,   # aggiungi nuovo batch
    lambda a, b: a - b,   # rimuovi batch uscito dalla finestra
    windowDuration=15,    # finestra di 15 secondi
    slideDuration=5       # slide ogni 5 secondi
)

def print_window_stats(rdd):
    if rdd.isEmpty():
        return
    top = rdd.takeOrdered(3, key=lambda x: -x[1])
    print(f"  [FINESTRA 15s] Top 3 stazioni:")
    for station, count in top:
        print(f"    → {station[:40]}: {count} trip")

windowed.foreachRDD(print_window_stats)

  [producer] chunk 03 scritto → 2000 righe


In [31]:
print("Avvio Spark Streaming... (durata ~90 secondi)")
print("="*60)

ssc.start()
ssc.awaitTerminationOrTimeout(90)  # gira per 90 secondi poi si ferma
ssc.stop(stopSparkContext=False, stopGraceFully=True)

print("\nStreaming completato.")

Avvio Spark Streaming... (durata ~90 secondi)

  [16:27:35] Batch — 1126 trip processati
    Wabash Ave & Grand Ave                31  ███
    McClurg Ct & Ohio St                  27  ██
    Millennium Park                       25  ██
    Clinton St & Jackson Blvd             18  █
    Broadway & Cornelia Ave               17  █
  [FINESTRA 15s] Top 3 stazioni:
    → Wabash Ave & Grand Ave: 31 trip
    → McClurg Ct & Ohio St: 27 trip
    → Millennium Park: 25 trip
  [producer] chunk 04 scritto → 2000 righe

  [16:27:40] Batch — 2000 trip processati
    Halsted St & Fulton St                55  █████
    Desplaines St & Kinzie St             46  ████
    Clark St & Elm St                     41  ████
    Canal St & Madison St                 37  ███
    Clark St & Schiller St                36  ███
  [FINESTRA 15s] Top 3 stazioni:
    → Halsted St & Fulton St: 57 trip
    → Clark St & Elm St: 55 trip
    → Desplaines St & Kinzie St: 52 trip
  [producer] chunk 05 scritto → 2000 righe



  [FINESTRA 15s] Top 3 stazioni:
    → Clark St & Elm St: 165 trip
    → Wabash Ave & Grand Ave: 147 trip
    → Dearborn Pkwy & Delaware Pl: 114 trip
  [producer] chunk 06 scritto → 2000 righe

  [16:27:50] Batch — 2000 trip processati
    Columbus Dr & Randolph St            136  █████████████
    Wells St & Elm St                     77  ███████
    Orleans St & Merchandise Mart Plaza   69  ██████
    State St & 33rd St                    61  ██████
    Clark St & Chicago Ave                55  █████


  [FINESTRA 15s] Top 3 stazioni:
    → Columbus Dr & Randolph St: 161 trip
    → Clark St & Elm St: 158 trip
    → Wabash Ave & Grand Ave: 120 trip
  [producer] chunk 07 scritto → 2000 righe

  [16:27:55] Batch — 2000 trip processati
    Sheffield Ave & Fullerton Ave         54  █████
    Desplaines St & Kinzie St             50  █████
    Benson Ave & Church St                45  ████
    Clark St & Schiller St                40  ████
    State St & 33rd St                    38  ███


  [FINESTRA 15s] Top 3 stazioni:
    → Columbus Dr & Randolph St: 159 trip
    → Wabash Ave & Grand Ave: 132 trip
    → Clark St & Elm St: 123 trip
  [producer] chunk 08 scritto → 2000 righe

  [16:28:00] Batch — 2000 trip processati
    Halsted St & Fulton St                62  ██████
    Ellis Ave & 60th St                   54  █████
    Halsted St & Polk St                  43  ████
    Desplaines St & Kinzie St             38  ███
    Clark St & Armitage Ave               36  ███


  [FINESTRA 15s] Top 3 stazioni:
    → Columbus Dr & Randolph St: 174 trip
    → Wells St & Elm St: 124 trip
    → Desplaines St & Kinzie St: 102 trip
  [producer] chunk 09 scritto → 2000 righe

  [16:28:05] Batch — 2000 trip processati
    Broadway & Cornelia Ave               81  ████████
    Michigan Ave & Lake St                69  ██████
    Halsted St & Polk St                  69  ██████
    Wabash Ave & Wacker Pl                61  ██████
    Morgan St & Lake St*                  41  ████


...

  [FINESTRA 15s] Top 3 stazioni:
    → Millennium Park: 208 trip
    → State St & 33rd St: 142 trip
    → Desplaines St & Kinzie St: 113 trip
  [producer] chunk 14 scritto → 2000 righe

  [16:28:30] Batch — 2000 trip processati
    State St & 33rd St                    87  ████████
    Clark St & Armitage Ave               56  █████
    Millennium Park                       41  ████
    Halsted St & Fulton St                37  ███
    Michigan Ave & Oak St                 37  ███


  [FINESTRA 15s] Top 3 stazioni:
    → Desplaines St & Kinzie St: 112 trip
    → Millennium Park: 101 trip
    → State St & 33rd St: 97 trip
  [producer] chunk 15 scritto → 2000 righe

  [16:28:35] Batch — 2000 trip processati
    Desplaines St & Kinzie St             60  ██████
    Calumet Ave & 33rd St                 55  █████
    Michigan Ave & Jackson Blvd           43  ████
    State St & Randolph St                36  ███
    Millennium Park                       35  ███


  [FINESTRA 15s] Top 3 stazioni:
    → Millennium Park: 127 trip
    → Desplaines St & Kinzie St: 99 trip
    → State St & 33rd St: 99 trip
  [producer] completato.
  [batch] nessun dato


  [FINESTRA 15s] Top 3 stazioni:
    → State St & 33rd St: 94 trip
    → Millennium Park: 76 trip
    → Desplaines St & Kinzie St: 73 trip
  [batch] nessun dato


  [FINESTRA 15s] Top 3 stazioni:
    → Desplaines St & Kinzie St: 60 trip
    → Calumet Ave & 33rd St: 55 trip
    → Michigan Ave & Jackson Blvd: 43 trip
  [batch] nessun dato
  [batch] nessun dato
  [batch] nessun dato
  [batch] nessun dato
  [batch] nessun dato

Streaming completato.


Per pulire le cartelle di streaming

In [67]:
shutil.rmtree(STREAM_DIR,  ignore_errors=True)
shutil.rmtree(CHECKPOINT,  ignore_errors=True)
print("Directory temporanee rimosse.")

Directory temporanee rimosse.


## Nuovo framework (DataFrame API)

Oltre alla DStream API, Spark offre una seconda API per lo streaming chiamata **Structured Streaming**, oggi considerata l'approccio standard per l'elaborazione di flussi di dati.

In Structured Streaming il flusso viene modellato come una tabella (DataFrame) in continua crescita. Ogni nuovo micro-batch aggiunge nuove righe alla tabella e Spark esegue automaticamente le query incrementali necessarie per aggiornare il risultato.

### Differenza rispetto a DStream

Nella DStream API uno stream è rappresentato come una sequenza di RDD:

```text
DStream = {RDD0, RDD1, RDD2, ...}

In Structured Streaming invece il flusso viene visto come una tabella in continua espansione:

          +-------------------+
tempo t0  | dati batch 0      |
          +-------------------+

tempo t1  | dati batch 0      |
          | dati batch 1      |
          +-------------------+

tempo t2  | dati batch 0      |
          | dati batch 1      |
          | dati batch 2      |
          +-------------------+


Lancio di una query cumulativa
- cumula i risultati a partire dal primo batch letto

In [ ]:
############################################################
## Parte 3 — Structured Streaming (DataFrame API)
##  UNA sola query cumulativa
############################################################

from pyspark.sql.types import *
from pyspark.sql.functions import col


# 1.  definizione dello schema CSV

stream_schema = StructType([
    StructField("_c0", StringType(), True),
    StructField("_c1", StringType(), True),
    StructField("_c2", StringType(), True),
    StructField("_c3", StringType(), True),
    StructField("_c4", StringType(), True),   # station
    StructField("_c5", StringType(), True),
    StructField("_c6", StringType(), True),
    StructField("_c7", StringType(), True),
    StructField("_c8", DoubleType(), True),   # lat
    StructField("_c9", DoubleType(), True),   # lng
])

# 2. Lettura streaming dalla directory
stream_df = (
    spark.readStream
         .option("header", "false")
         .schema(stream_schema)
         .csv(STREAM_DIR)
)

# 3. Filtro geografico + pulizia dati

filtered_df = (
    stream_df
    .filter(
        (col("_c8") >= 41.6) &
        (col("_c8") <= 42.1) &
        (col("_c9") >= -87.9) &
        (col("_c9") <= -87.5)
    )
    .filter(col("_c4").isNotNull())
)

# 4. Aggregazione cumulativa (totale per stazione ovvero il campo quarto)
trip_counts_df = (
    filtered_df
    .groupBy("_c4")
    .count()
)

# ----------------------------------------------------------
# 5. UNA  QUERY STREAMING  (agregazione cumulativa, output console)
# ----------------------------------------------------------
query = (
    trip_counts_df
    .orderBy(col("count").desc())
    .writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .start()
)



  [producer] chunk 03 scritto → 2000 righe


In [80]:
# ----------------------------------------------------------
# 6. Avvio streaming
# ----------------------------------------------------------
print("Structured Streaming avviato ...")
print("=" * 60)

query.awaitTermination(90)

query.stop()

print("Streaming completato.")

Structured Streaming avviato ...


  [producer] chunk 04 scritto → 2000 righe


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|California Ave & Milwaukee Ave       |16   |
|Wood St & Chicago Ave                |8    |
|Campbell Ave & Montrose Ave          |8    |
|Morgan Ave & 14th Pl                 |7    |
|California Ave & Division St         |4    |
|Kedzie Ave & Lake St                 |3    |
|Western Ave & Roscoe St              |2    |
|Chicago State University             |2    |
|Public Rack - Michigan Ave & 119th St|1    |
+-------------------------------------+-----+



  [producer] chunk 05 scritto → 2000 righe


-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------------------+-----+
|_c4                          |count|
+-----------------------------+-----+
|Halsted St & Fulton St       |57   |
|Clark St & Elm St            |55   |
|Desplaines St & Kinzie St    |52   |
|Clark St & Schiller St       |47   |
|Canal St & Madison St        |43   |
|McClurg Ct & Ohio St         |42   |
|Wabash Ave & Grand Ave       |41   |
|Millennium Park              |40   |
|Clark St & Armitage Ave      |37   |
|Elizabeth St & Randolph St   |33   |
|Broadway & Cornelia Ave      |32   |
|Wabash Ave & 9th St          |31   |
|Sheffield Ave & Fullerton Ave|29   |
|Michigan Ave & 8th St        |29   |
|Wells St & Elm St            |29   |
|Larrabee St & Kingsbury St   |29   |
|Dearborn Pkwy & Delaware Pl  |28   |
|Sheffield Ave & Webster Ave  |27   |
|Canal St & Adams St          |27   |
|Wells St & Concord Ln        |27   |
+----------------------------

...

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|Halsted St & Fulton St               |269  |
|State St & 33rd St                   |248  |
|Michigan Ave & 8th St                |239  |
|Columbus Dr & Randolph St            |226  |
|Halsted St & Polk St                 |217  |
|Millennium Park                      |212  |
|Clark St & Elm St                    |208  |
|Desplaines St & Kinzie St            |193  |
|Green St & Madison St                |192  |
|Larrabee St & Kingsbury St           |189  |
|Damen Ave & Pierce Ave               |186  |
|Dearborn Pkwy & Delaware Pl          |183  |
|Wabash Ave & Grand Ave               |182  |
|McClurg Ct & Ohio St                 |181  |
|Broadway & Cornelia Ave              |177  |
|Milwaukee Ave & Grand Ave            |176  |
|Peoria St & Jackson Blvd    

  [producer] chunk 14 scritto → 2000 righe


-------------------------------------------
Batch: 7
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|Halsted St & Fulton St               |326  |
|Desplaines St & Kinzie St            |292  |
|Columbus Dr & Randolph St            |276  |
|Millennium Park                      |272  |
|State St & 33rd St                   |258  |
|Michigan Ave & 8th St                |251  |
|Clark St & Elm St                    |251  |
|Wabash Ave & Grand Ave               |243  |
|Dearborn Pkwy & Delaware Pl          |237  |
|Halsted St & Polk St                 |236  |
|Wells St & Elm St                    |224  |
|McClurg Ct & Ohio St                 |220  |
|DuSable Lake Shore Dr & Diversey Pkwy|216  |
|Sheffield Ave & Fullerton Ave        |203  |
|Green St & Madison St                |202  |
|Larrabee St & Kingsbury St           |195  |
|Federal St & Polk St        

  [producer] completato.


-------------------------------------------
Batch: 8
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|Halsted St & Fulton St               |386  |
|Desplaines St & Kinzie St            |365  |
|State St & 33rd St                   |352  |
|Millennium Park                      |348  |
|Columbus Dr & Randolph St            |312  |
|Michigan Ave & 8th St                |295  |
|Wabash Ave & Grand Ave               |286  |
|Clark St & Elm St                    |278  |
|Halsted St & Polk St                 |275  |
|Dearborn Pkwy & Delaware Pl          |256  |
|Wells St & Elm St                    |247  |
|McClurg Ct & Ohio St                 |243  |
|Sheffield Ave & Fullerton Ave        |241  |
|Green St & Madison St                |237  |
|Clark St & Schiller St               |228  |
|Clark St & Armitage Ave              |227  |
|DuSable Lake Shore Dr & Dive

In [55]:
# ----------------------------------------------------------
# 7. Avvio e attesa
# ----------------------------------------------------------
print("Structured Streaming avviato...")
print("=" * 60)

query_simple.awaitTermination(90)
query_window.awaitTermination(90)

query_simple.stop()
query_window.stop()

print("Structured Streaming completato.")

Structured Streaming avviato...


  [producer] chunk 05 scritto → 2000 righe


  [producer] chunk 06 scritto → 2000 righe


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|California Ave & Milwaukee Ave       |11   |
|Wood St & Chicago Ave                |5    |
|Morgan Ave & 14th Pl                 |4    |
|Kedzie Ave & Lake St                 |3    |
|Campbell Ave & Montrose Ave          |3    |
|California Ave & Division St         |2    |
|Chicago State University             |2    |
|Public Rack - Michigan Ave & 119th St|1    |
+-------------------------------------+-----+



  [producer] chunk 07 scritto → 2000 righe


...

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|Halsted St & Fulton St               |363  |
|State St & 33rd St                   |345  |
|Millennium Park                      |313  |
|Desplaines St & Kinzie St            |305  |
|Columbus Dr & Randolph St            |289  |
|Michigan Ave & 8th St                |284  |
|Clark St & Elm St                    |269  |
|Wabash Ave & Grand Ave               |263  |
|Halsted St & Polk St                 |263  |
|Dearborn Pkwy & Delaware Pl          |249  |
|Wells St & Elm St                    |236  |
|Sheffield Ave & Fullerton Ave        |228  |
|McClurg Ct & Ohio St                 |227  |
|Clark St & Armitage Ave              |222  |
|DuSable Lake Shore Dr & Diversey Pkwy|218  |
|Clark St & Schiller St               |218  |
|Green St & Madison St       

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-------------------------------------+-----+
|window                                    |_c4                                  |count|
+------------------------------------------+-------------------------------------+-----+
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Millennium Park                      |290  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|State St & 33rd St                   |244  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Halsted St & Fulton St               |239  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Desplaines St & Kinzie St            |208  |
|{2026-06-08 18:59:00, 2026-06-08 18:59:15}|Columbus Dr & Randolph St            |202  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Halsted St & Polk St                 |195  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Federal St & Polk St                 |191  |
|{2026-06-08 

-------------------------------------------
Batch: 3
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|Halsted St & Fulton St               |386  |
|Desplaines St & Kinzie St            |365  |
|State St & 33rd St                   |352  |
|Millennium Park                      |348  |
|Columbus Dr & Randolph St            |312  |
|Michigan Ave & 8th St                |295  |
|Wabash Ave & Grand Ave               |286  |
|Clark St & Elm St                    |278  |
|Halsted St & Polk St                 |275  |
|Dearborn Pkwy & Delaware Pl          |256  |
|Wells St & Elm St                    |247  |
|McClurg Ct & Ohio St                 |243  |
|Sheffield Ave & Fullerton Ave        |241  |
|Green St & Madison St                |237  |
|Clark St & Schiller St               |228  |
|Clark St & Armitage Ave              |227  |
|DuSable Lake Shore Dr & Dive

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+-------------------------------------+-----+
|window                                    |_c4                                  |count|
+------------------------------------------+-------------------------------------+-----+
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Halsted St & Fulton St               |241  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Millennium Park                      |225  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Halsted St & Polk St                 |199  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|DuSable Lake Shore Dr & Diversey Pkwy|189  |
|{2026-06-08 18:59:00, 2026-06-08 18:59:15}|Columbus Dr & Randolph St            |180  |
|{2026-06-08 18:59:00, 2026-06-08 18:59:15}|Clark St & Elm St                    |178  |
|{2026-06-08 18:59:30, 2026-06-08 18:59:45}|Federal St & Polk St                 |173  |
|{2026-06-08 

-------------------------------------------
Batch: 3
-------------------------------------------
+-------------------------------------+-----+
|_c4                                  |count|
+-------------------------------------+-----+
|Halsted St & Fulton St               |386  |
|Desplaines St & Kinzie St            |365  |
|State St & 33rd St                   |352  |
|Millennium Park                      |348  |
|Columbus Dr & Randolph St            |312  |
|Michigan Ave & 8th St                |295  |
|Wabash Ave & Grand Ave               |286  |
|Clark St & Elm St                    |278  |
|Halsted St & Polk St                 |275  |
|Dearborn Pkwy & Delaware Pl          |256  |
|Wells St & Elm St                    |247  |
|McClurg Ct & Ohio St                 |243  |
|Sheffield Ave & Fullerton Ave        |241  |
|Green St & Madison St                |237  |
|Clark St & Schiller St               |228  |
|Clark St & Armitage Ave              |227  |
|DuSable Lake Shore Dr & Dive

In [74]:
shutil.rmtree(STREAM_DIR,  ignore_errors=True)
shutil.rmtree(CHECKPOINT,  ignore_errors=True)
print("Directory temporanee rimosse.")

Directory temporanee rimosse.



I due output ottenuti non coincidono perfettamente perché:

#### DStream
Nel caso DStream ogni micro-batch è trattato come un insieme indipendente di dati. <br>Le aggregazioni sono costruite esplicitamente a partire dai batch ricevuti e la finestra temporale è ottenuta combinando manualmente i risultati dei batch precedenti  (agreggazione di 15 secondi a finestra).


#### Structured Streaming
Nel caso Structured Streaming Spark mantiene internamente lo stato delle aggregazioni e gestisce in modo automatico l’evoluzione dei risultati nel tempo.<br>
In questo caso la query  ottiene una aggregazione progressiva a partire dal primo batch.
